In [0]:
%pip install faker

In [0]:
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta
import pandas as pd

fake = Faker()
Faker.seed(42)
random.seed(42)

# --- City mapping ---
country_city_map = {
    "India": ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai"],
    "United States": ["New York", "Los Angeles", "Chicago", "Houston", "San Francisco"],
    "United Kingdom": ["London", "Manchester", "Birmingham", "Liverpool", "Leeds"],
    "Australia": ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"],
    "Canada": ["Toronto", "Vancouver", "Montreal", "Calgary", "Ottawa"]
}

def get_city_for_country(country):
    if country in country_city_map:
        return random.choice(country_city_map[country])
    else:
        return fake.city()

# --- Dataset 2: customer_profile.csv ---
num_customers = random.randint(300, 500)
customer_ids = [str(uuid.uuid4()) for _ in range(num_customers)]
account_ids = [str(uuid.uuid4()) for _ in range(num_customers)]
customer_profiles = []

for i in range(num_customers):
    home_country = "India" if random.random() < 0.92 else random.choice(list(country_city_map.keys()))
    home_city = get_city_for_country(home_country)
    kyc_status = "VERIFIED" if random.random() < 0.93 else "PENDING"
    is_active = True if random.random() < 0.96 else False
    account_type = random.choice(["SAVINGS", "CURRENT"])
    dob = fake.date_of_birth(minimum_age=18, maximum_age=70)
    opening_date = fake.date_between(start_date="-10y", end_date="-1d")
    account_balance = round(random.uniform(1000, 1000000), 2)
    customer_profiles.append({
        "customer_id": customer_ids[i],
        "account_id": account_ids[i],
        "customer_name": fake.name(),
        "email": fake.email(),
        "phone": fake.phone_number(),
        "date_of_birth": dob,
        "kyc_status": kyc_status,
        "home_country": home_country,
        "home_city": home_city,
        "account_opening_date": opening_date,
        "account_type": account_type,
        "account_balance": account_balance,
        "is_active": is_active,
        "_ingest_ts": datetime.now()
    })

customer_profile_df = pd.DataFrame(customer_profiles)
customer_profile_df.to_csv("customer_profile.csv", index=False)

# --- Dataset 3: device_sessions.csv ---
device_types = ["Mobile", "Laptop", "Tablet", "Desktop"]
oses = ["Android", "iOS", "Windows", "MacOS", "Linux"]
browsers = ["Chrome", "Safari", "Firefox", "Edge", "Opera"]
num_sessions = 1000
device_sessions = []

now = datetime.now()
start_range = now - timedelta(days=150)  # 5 months ago

for _ in range(num_sessions):
    cust_idx = random.randint(0, num_customers - 1)
    customer_id = customer_ids[cust_idx]
    device_id = str(uuid.uuid4())
    session_id = str(uuid.uuid4())
    login_ts = fake.date_time_between_dates(datetime_start=start_range, datetime_end=now - timedelta(minutes=5))
    session_duration = random.randint(5, 480)
    logout_ts = login_ts + timedelta(minutes=session_duration)
    if logout_ts > now:
        logout_ts = now
    login_ts_str = login_ts.strftime("%Y-%m-%d %H:%M:%S")
    logout_ts_str = logout_ts.strftime("%Y-%m-%d %H:%M:%S")
    ip = fake.ipv4()
    country = customer_profile_df.loc[cust_idx, "home_country"]
    city = customer_profile_df.loc[cust_idx, "home_city"]
    device_type = random.choice(device_types)
    os = random.choice(oses)
    browser = random.choice(browsers)
    is_new_device = random.random() < 0.08
    is_new_location = random.random() < 0.07
    if is_new_location:
        country = random.choice(list(country_city_map.keys()))
        city = get_city_for_country(country)
    device_sessions.append({
        "session_id": session_id,
        "customer_id": customer_id,
        "device_id": device_id,
        "login_timestamp": login_ts_str,
        "logout_timestamp": logout_ts_str,
        "ip_address": ip,
        "location_country": country,
        "location_city": city,
        "device_type": device_type,
        "os": os,
        "browser": browser,
        "is_new_device": is_new_device,
        "is_new_location": is_new_location,
        "_ingest_ts": datetime.now()
    })

device_sessions_df = pd.DataFrame(device_sessions)
device_sessions_df.to_csv("device_sessions.csv", index=False)

# --- Dataset 1: transactions.csv ---
txn_types = ["WITHDRAWAL", "DEBIT", "CREDIT", "TRANSFER", "DEPOSIT"]
channels_map = {
    "WITHDRAWAL": ["ATM", "BRANCH"],
    "DEBIT": ["UPI", "ONLINE", "POS"],
    "CREDIT": ["ONLINE", "MOBILE", "BRANCH"],
    "TRANSFER": ["ONLINE", "MOBILE", "UPI"],
    "DEPOSIT": ["BRANCH", "ATM"]
}
statuses = ["SUCCESS", "FAILED", "PENDING"]
merchant_categories = ["GROCERY", "ELECTRONICS", "RESTAURANT", "TRAVEL", "FASHION", "UTILITY", "FUEL"]
currencies = ["INR", "USD", "EUR"]
transactions = []

for idx, row in customer_profile_df.iterrows():
    customer_id = row["customer_id"]
    acc_id = row["account_id"]
    account_balance = row["account_balance"]
    account_type = row["account_type"]
    # Determine number of transactions based on balance
    if account_balance < 5000:
        num_txns = random.randint(1, 3)
        max_txn_amt = account_balance * 0.5
    elif account_balance < 50000:
        num_txns = random.randint(2, 8)
        max_txn_amt = account_balance * 0.6
    elif account_balance < 200000:
        num_txns = random.randint(5, 15)
        max_txn_amt = account_balance * 0.7
    else:
        num_txns = random.randint(10, 25)
        max_txn_amt = account_balance * 0.8

    total_txn_amt = 0
    for _ in range(num_txns):
        txn_type = random.choice(txn_types)
        channel = random.choice(channels_map[txn_type])
        merchant_id = str(uuid.uuid4())
        merchant_category = random.choice(merchant_categories)
        device_ids_for_customer = device_sessions_df[device_sessions_df.customer_id == customer_id]["device_id"].tolist()
        device_id = random.choice(device_ids_for_customer) if device_ids_for_customer else str(uuid.uuid4())
        txn_time = fake.date_time_between(start_date="-30d", end_date="now")
        # Amount logic: proportional, never exceeds balance
        remaining_balance = account_balance - total_txn_amt
        if remaining_balance <= 0:
            break
        if txn_type == "WITHDRAWAL":
            amount = round(random.uniform(100, min(max_txn_amt, remaining_balance)), 2)
            from_account = acc_id
            to_account = None
        elif txn_type == "DEPOSIT":
            amount = round(random.uniform(100, min(max_txn_amt, remaining_balance)), 2)
            from_account = None
            to_account = acc_id
        elif txn_type == "TRANSFER":
            amount = round(random.uniform(100, min(max_txn_amt, remaining_balance)), 2)
            from_account = acc_id
            to_account = str(uuid.uuid4())
        elif txn_type == "DEBIT":
            amount = round(random.uniform(100, min(max_txn_amt, remaining_balance)), 2)
            from_account = acc_id
            to_account = None
        elif txn_type == "CREDIT":
            amount = round(random.uniform(100, min(max_txn_amt, remaining_balance)), 2)
            from_account = None
            to_account = acc_id
        else:
            amount =
    status = "SUCCESS" if random.random() < 0.93 else random.choice(["FAILED", "PENDING"])
    txn_type = random.choice(txn_types)
    channel = random.choice(channels)
    merchant_id = str(uuid.uuid4())
    merchant_category = random.choice(merchant_categories)
    country = customer_profile_df.loc[cust_idx, "home_country"]
    city = customer_profile_df.loc[cust_idx, "home_city"]
    # Some fraud: different country, new device
    if random.random() < 0.02:
        country = fake.country()
        city = fake.city()
        device_id = str(uuid.uuid4())
    transactions.append({
        "txn_id": txn_id,
        "account_id": acc_id,
        "customer_id": customer_id,
        "txn_timestamp": txn_time,
        "txn_type": txn_type,
        "amount": round(amount, 2),
        "currency": random.choice(currencies),
        "from_account": acc_id,
        "to_account": str(uuid.uuid4()),
        "channel": channel,
        "device_id": device_id,
        "ip_address": fake.ipv4(),
        "location_country": country,
        "location_city": city,
        "location_lat": fake.latitude(),
        "location_lon": fake.longitude(),
        "merchant_id": merchant_id,
        "merchant_category": merchant_category,
        "status": status,
        "_ingest_ts": datetime.now(),
        "_rescued_data": ""
    })

# --- Messy Data: 5% imperfect ---
messy_count = int(0.05 * num_txns)
for i in range(messy_count):
    idx = random.randint(0, len(transactions) - 1)
    txn = transactions[idx]
    # Duplicate txn_id
    if random.random() < 0.3:
        transactions[idx]["txn_id"] = random.choice(list(fraud_txn_ids))
    # Negative amount
    if random.random() < 0.3:
        transactions[idx]["amount"] = -abs(transactions[idx]["amount"])
    # Inconsistent txn_type
    if random.random() < 0.3:
        transactions[idx]["txn_type"] = random.choice(txn_types_inconsistent)
    # Null values
    if random.random() < 0.3:
        col = random.choice([
            "location_country", "location_city", "device_id", "merchant_id", "merchant_category"
        ])
        transactions[idx][col] = None
    # Incorrect locations
    if random.random() < 0.3:
        transactions[idx]["location_country"] = "N/A"
        transactions[idx]["location_city"] = "Unknown"

transactions_df = pd.DataFrame(transactions)
transactions_df.to_csv("transactions.csv", index=False)

In [0]:
customer_profile_df.to_csv("customer_profile_table1.csv", index=False)

device_sessions_df.to_csv("device_sessions_table1.csv", index=False)

transactions_df.to_csv("transactions_table1.csv", index=False)